In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
path = "datasets/global_bleaching_environmental.csv"
df = pd.read_csv(path, low_memory=False, na_values=["nd", "ND", ""])

In [4]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = [
    "Percent_Bleaching", "Temperature_Mean", "SSTA", "TSA",
    "Depth_m", "Latitude_Degrees", "Longitude_Degrees", "Cyclone_Frequency"
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.shape)
display(df.head())
display(df[numeric_cols].describe())

(41361, 62)


,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,...,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Site_Comments,Sample_Comments,Bleaching_Comments
0,2501,10324336,Donner,23.163,-82.5260,Atlantic,NaN,Tropical Atlantic,Cuba and Cayman Islands,Cuba,...,5.0,0.0,0.00,0.74,7.25,0.18,2005-09-15,NaN,NaN,NaN
1,3467,10324754,Donner,-17.575,-149.7833,Pacific,NaN,Eastern Indo-Pacific,Society Islands French Polynesia,French Polynesia,...,4.0,0.0,0.26,0.67,4.65,0.19,1991-03-15,The bleaching does not appear to have gained ...,The bleaching does not appear to have gained ...,NaN
2,1794,10323866,Donner,18.369,-64.5640,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United Kingdom,...,7.0,0.0,0.00,1.04,11.66,0.26,2006-01-15,NaN,NaN,NaN
3,8647,10328028,Donner,17.760,-64.5680,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,4.0,0.0,0.00,0.75,5.64,0.20,2006-04-15,NaN,NaN,NaN
4,8648,10328029,Donner,17.769,-64.5830,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,5.0,0.0,0.00,0.92,6.89,0.25,2006-04-15,NaN,NaN,NaN


,Percent_Bleaching,Temperature_Mean,SSTA,TSA,Depth_m,Latitude_Degrees,Longitude_Degrees,Cyclone_Frequency
count,34515.000000,41229.000000,41213.000000,41213.000000,39562.000000,41361.000000,41361.000000,41361.000000
mean,9.619465,300.410472,0.254942,-0.980981,6.922119,7.558085,34.966127,52.159650
std,20.190956,1.578528,0.829508,1.640874,4.162464,15.732185,103.404598,7.589593
min,0.000000,290.880000,-4.620000,-11.970000,0.000000,-30.262500,-179.974300,18.310000
25%,0.000000,299.740000,-0.250000,-1.810000,3.700000,-4.902500,-78.385600,47.940000
50%,0.250000,300.760000,0.240000,-0.740000,6.000000,10.776100,96.843300,50.920000
75%,6.000000,301.640000,0.750000,0.100000,10.000000,20.050500,120.880400,55.730000
max,100.000000,303.520000,5.900000,5.900000,90.000000,36.750000,179.964500,105.800000


In [5]:
gcbd = df[df["Percent_Bleaching"].notna()]

In [6]:
import requests
import pandas as pd
import os
import time
import xarray as xr
from io import BytesIO

# gcbd = pd.read_csv("global_bleaching_environmental.csv")
unique_sites = gcbd[['Latitude_Degrees','Longitude_Degrees']].drop_duplicates().reset_index(drop=True)
print(f"Unique sites: {len(unique_sites)}")

os.makedirs("cortad_sites", exist_ok=True)

base = "https://www.ncei.noaa.gov/thredds-ocean/ncss/cortad/Version6"

ok = 0
fail = 0
skip = 0

for i, row in unique_sites.iterrows():
    lat = row['Latitude_Degrees']
    lon = row['Longitude_Degrees']
    
    outfile = f"cortad_sites/site_{i:05d}.csv"
    if os.path.exists(outfile):
        skip += 1
        continue
    
    # NCSS point request for FilledSST
    sst_params = {
        "var": "FilledSST",
        "latitude": lat,
        "longitude": lon,
        "time_start": "1982-01-02T00:00:00Z",
        "time_end": "2022-12-30T00:00:00Z",
        "accept": "csv"
    }
    
    tsa_params_list = []
    for var in ["TSA", "TSA_DHW", "TSA_Frequency"]:
        tsa_params_list.append({
            "var": var,
            "latitude": lat,
            "longitude": lon,
            "time_start": "1982-01-02T00:00:00Z",
            "time_end": "2022-12-30T00:00:00Z",
            "accept": "csv"
        })
    
    try:
        # Fetch SST
        sst_resp = requests.get(
            f"{base}/cortadv6_FilledSST.nc",
            params=sst_params, timeout=120
        )
        sst_resp.raise_for_status()
        sst_df = pd.read_csv(BytesIO(sst_resp.content))
        
        # Fetch TSA variables
        for tp in tsa_params_list:
            var_name = tp["var"]
            resp = requests.get(
                f"{base}/cortadv6_TSA.nc",
                params=tp, timeout=120
            )
            resp.raise_for_status()
            var_df = pd.read_csv(BytesIO(resp.content))
            sst_df[var_name] = var_df[var_name] if var_name in var_df.columns else None
        
        sst_df['site_lat'] = lat
        sst_df['site_lon'] = lon
        sst_df.to_csv(outfile, index=False)
        
        ok += 1
        if ok % 10 == 0:
            total = ok + fail + skip
            print(f"Progress: {total}/{len(unique_sites)} | OK:{ok} Fail:{fail} Skip:{skip}")
    
    except Exception as e:
        fail += 1
        if fail <= 5:
            print(f"Site {i} ({lat}, {lon}): {type(e).__name__}: {e}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"  Response: {e.response.text[:200]}")
    
    time.sleep(0.5)

print(f"\nDone! OK:{ok} Fail:{fail} Skip:{skip}")

Unique sites: 10847
Site 0 (23.163, -82.526): HTTPError: 404 Client Error:  for url: https://www.ncei.noaa.gov/thredds-ocean/ncss/cortad/Version6/cortadv6_FilledSST.nc?var=FilledSST&latitude=23.163&longitude=-82.526&time_start=1982-01-02T00%3A00%3A00Z&time_end=2022-12-30T00%3A00%3A00Z&accept=csv
  Response: <!doctype html><html lang="en"><head><title>HTTP Status 404 – Not Found</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} 
Site 1 (-17.575, -149.7833): HTTPError: 404 Client Error:  for url: https://www.ncei.noaa.gov/thredds-ocean/ncss/cortad/Version6/cortadv6_FilledSST.nc?var=FilledSST&latitude=-17.575&longitude=-149.7833&time_start=1982-01-02T00%3A00%3A00Z&time_end=2022-12-30T00%3A00%3A00Z&accept=csv
  Response: <!doctype html><html lang="en"><head><title>HTTP Status 404 – Not Found</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D

KeyboardInterrupt: 